# `trainer_v2.ipynb` — 3-way split, early-stopped fine-tune (up to 50 epochs)

End-to-end fine-tune of BiRefNet on the Hypersim room data, entirely in this notebook, with proper
methodology:

- **Train / Val / Test split — by SCENE (70/15/15)** so frames of one room never cross splits.
- **Phase A — fine-tune on TRAIN**, monitor **VAL** every epoch, **early stop** (patience) on val mean-IoU,
  keep the **best** checkpoint. Max 50 epochs.
- **Phase B — fold VAL into training** and fine-tune the best model a few more epochs
  ("use validation data for fine-tuning" once it has served model-selection).
- **Final — evaluate on the untouched TEST set** (metrics + confusion matrix). Test is only ever looked at once.

Checkpoints + `history.json` are written to `hq-mat/BiRefNet/ckpts/hypersim_es/` after every epoch, so
progress survives interruption.

> Faithful to the original training: same model (swin_v1_l, bs=1 → BN=Identity), same `PixLoss`
> (BCE+IoU+SSIM+MAE, multi-scale), lr 5e-6, bf16. See `trainer.ipynb` for the concepts.

## 0. Setup

In [ ]:
import os, sys, re, json, glob, random, time
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path("/home/krist/works/refine-mask"); REPO = ROOT/"hq-mat"/"BiRefNet"
sys.path.insert(0, str(REPO)); os.chdir(REPO)
torch.set_float32_matmul_precision("high")
SEED = 7; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# build the model at batch_size=1 (BN -> Identity) so it matches our checkpoints
import models.birefnet as _bn
from models.modules import aspp as _aspp, decoder_blocks as _dec
_aspp.config.batch_size = 1; _dec.config.batch_size = 1
class _BS1(_bn.Config):
    def __init__(self, *a, **k): super().__init__(*a, **k); self.batch_size = 1
_bn.Config = _BS1
from config import Config
from loss import PixLoss
cfg = Config()

SIZE = 640
OUT = REPO/"ckpts/hypersim_es"; OUT.mkdir(parents=True, exist_ok=True)
# hyperparameters
MAX_EPOCHS = 50
PATIENCE   = 7          # stop if val mean-IoU doesn't improve for this many epochs
MIN_DELTA  = 1e-3
LR         = 5e-6
PHASE_B_EPOCHS = 3      # short fine-tune on train+val after early stop
print("device", torch.cuda.get_device_name(0), "| out_ref", cfg.out_ref)


## 1. Train / Val / Test split by scene

The extracted images already exist (`HypersimRooms` + `hypersim_val`); each filename starts with its scene
id `ai_XXX_YYY`. We pool them all, group by scene, and split the **scenes** 70/15/15. No files are moved.

In [ ]:
pools = [Path.home()/"datasets/dis/General/HypersimRooms",
         Path.home()/"datasets/dis/hypersim_val"]
pairs = []   # (img_path, gt_path)
for base in pools:
    for ip in (base/"im").glob("*.jpg"):
        gp = base/"gt"/(ip.stem+".png")
        if gp.exists(): pairs.append((ip, gp))
def scene_of(p): return re.match(r"(ai_\d+_\d+)", p.stem).group(1)
scenes = sorted({scene_of(ip) for ip, _ in pairs})
random.Random(SEED).shuffle(scenes)
n = len(scenes); n_tr = int(n*0.70); n_va = int(n*0.15)
tr_s, va_s, te_s = set(scenes[:n_tr]), set(scenes[n_tr:n_tr+n_va]), set(scenes[n_tr+n_va:])
split = {"train": [], "val": [], "test": []}
for ip, gp in pairs:
    s = scene_of(ip)
    k = "train" if s in tr_s else "val" if s in va_s else "test"
    split[k].append((str(ip), str(gp)))
json.dump({"scenes": {"train": sorted(tr_s), "val": sorted(va_s), "test": sorted(te_s)}},
          open(OUT/"split_scenes.json", "w"), indent=1)
print(f"scenes {n}  -> train {len(tr_s)} / val {len(va_s)} / test {len(te_s)}")
print(f"imgs      -> train {len(split['train'])} / val {len(split['val'])} / test {len(split['test'])}")


## 2. Dataset & loaders

In [ ]:
class RoomDS(torch.utils.data.Dataset):
    def __init__(self, items, train=False):
        self.items = items; self.train = train
        self.norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        ip, gp = self.items[i]
        img = Image.open(ip).convert("RGB").resize((SIZE,SIZE), Image.BILINEAR)
        gt  = Image.open(gp).convert("L").resize((SIZE,SIZE), Image.NEAREST)
        if self.train and random.random() < 0.5:            # light aug: hflip
            img = img.transpose(Image.FLIP_LEFT_RIGHT); gt = gt.transpose(Image.FLIP_LEFT_RIGHT)
        x = self.norm(transforms.functional.to_tensor(img))
        y = transforms.functional.to_tensor(gt)
        return x, y

def loader(items, train):
    return torch.utils.data.DataLoader(RoomDS(items, train), batch_size=1, shuffle=train,
                                       num_workers=4, pin_memory=True, drop_last=train)
train_loader = loader(split["train"], True)
val_loader   = loader(split["val"],   False)
test_loader  = loader(split["test"],  False)
print("batches:", len(train_loader), len(val_loader), len(test_loader))


## 3. Train / eval helpers (faithful loss, fp32 outside autocast)

In [ ]:
pix_loss = PixLoss()

def train_one_epoch(model, opt, dl):
    model.train(); tot = 0.0
    for x, y in dl:
        x, y = x.cuda(), y.cuda()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            sp, _ = model(x)
            if cfg.out_ref: (_, _), sp = sp
        sp = [s.float() for s in sp]                    # BCE is autocast-unsafe -> fp32 loss
        loss, _ = pix_loss(sp, torch.clamp(y, 0, 1), pix_loss_lambda=1.0)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss)
    return tot/len(dl)

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); ious=[]; losses=[]
    for x, y in dl:
        x, y = x.cuda(), y.cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(x)[-1].sigmoid().float()
        p = pred.cpu()[0,0].numpy() > 0.5
        g = y.cpu()[0,0].numpy() > 0.5
        ious.append((p&g).sum()/((p|g).sum()+1e-6))
    return float(np.mean(ious))          # val mean-IoU (early-stop metric)


## 4. Phase A — fine-tune on TRAIN, early stop on VAL

Keeps the best checkpoint by val mean-IoU; stops after `PATIENCE` epochs without a `MIN_DELTA` improvement.

In [ ]:
from models.birefnet import BiRefNet
model = BiRefNet(bb_pretrained=False).cuda()
model.load_state_dict(torch.load(REPO/"pretrained_init/epoch_0.pth", weights_only=True))
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=1e-2)

history=[]; best_iou=-1; best_ep=0; wait=0
for ep in range(1, MAX_EPOCHS+1):
    t=time.time()
    tr = train_one_epoch(model, opt, train_loader)
    vi = evaluate(model, val_loader)
    history.append({"epoch":ep,"train_loss":round(tr,4),"val_iou":round(vi,4),"sec":round(time.time()-t,1)})
    json.dump(history, open(OUT/"history.json","w"), indent=1)
    improved = vi > best_iou + MIN_DELTA
    if improved:
        best_iou, best_ep, wait = vi, ep, 0
        torch.save(model.state_dict(), OUT/"best.pth")
    else:
        wait += 1
    print(f"ep {ep:2d}  train_loss {tr:8.2f}  val_IoU {vi:.4f}  {'*best*' if improved else f'(no-improve {wait}/{PATIENCE})'}")
    if wait >= PATIENCE:
        print(f"early stop at epoch {ep}; best epoch {best_ep} val_IoU {best_iou:.4f}")
        break
print("Phase A done. best epoch", best_ep, "val IoU", round(best_iou,4))


## 5. Curves — train loss & val IoU

In [ ]:
h=json.load(open(OUT/"history.json"))
eps=[r["epoch"] for r in h]
fig,ax=plt.subplots(1,2,figsize=(13,4))
ax[0].plot(eps,[r["train_loss"] for r in h],marker="o",ms=3); ax[0].set_title("Phase A train loss"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3)
ax[1].plot(eps,[r["val_iou"] for r in h],marker="o",ms=3,color="green")
ax[1].axvline(best_ep,ls="--",color="red",label=f"best ep {best_ep}"); ax[1].set_title("Val mean-IoU (early-stop metric)"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. Phase B — fold VAL into training, short fine-tune

Once val has chosen the best model, its labels are "free" data: continue training the best checkpoint on
**train + val** for a few epochs at half LR. Test stays untouched.

In [ ]:
model.load_state_dict(torch.load(OUT/"best.pth", weights_only=True))
tv_loader = loader(split["train"] + split["val"], True)
opt2 = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR*0.5, weight_decay=1e-2)
for ep in range(1, PHASE_B_EPOCHS+1):
    tr = train_one_epoch(model, opt2, tv_loader)
    print(f"[phase B] ep {ep}  train_loss {tr:8.2f}")
torch.save(model.state_dict(), OUT/"final.pth")
print("Phase B done -> final.pth")


## 7. Final evaluation on TEST (best vs final) + confusion matrix

In [ ]:
def full_metrics(model, items):
    model.eval(); TP=FP=FN=TN=0; ious=[]
    norm=transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    with torch.no_grad():
        for ip,gp in items:
            img=Image.open(ip).convert("RGB").resize((SIZE,SIZE),Image.BILINEAR)
            g=np.array(Image.open(gp).convert("L").resize((SIZE,SIZE),Image.NEAREST))>127
            x=norm(transforms.functional.to_tensor(img)).unsqueeze(0).cuda()
            with torch.autocast("cuda",dtype=torch.float16):
                p=model(x)[-1].sigmoid().float().cpu()[0,0].numpy()>0.5
            TP+=int((p&g).sum()); FP+=int((p&~g).sum()); FN+=int((~p&g).sum()); TN+=int((~p&~g).sum())
            ious.append((p&g).sum()/((p|g).sum()+1e-6))
    tot=TP+FP+FN+TN
    return dict(acc=(TP+TN)/tot, prec=TP/(TP+FP+1e-9), rec=TP/(TP+FN+1e-9),
               iou_pix=TP/(TP+FP+FN+1e-9), iou_mean=float(np.mean(ious)),
               cm=np.array([[TN,FP],[FN,TP]],float))

for name in ["best","final"]:
    model.load_state_dict(torch.load(OUT/f"{name}.pth", weights_only=True))
    m=full_metrics(model, split["test"])
    print(f"[TEST/{name}] acc={m['acc']:.4f} P={m['prec']:.4f} R={m['rec']:.4f} IoU_pix={m['iou_pix']:.4f} IoU_mean={m['iou_mean']:.4f}")
    if name=="final": M=m

cm=M["cm"]; cmn=cm/cm.sum(1,keepdims=True); labs=["BG (structure)","FG (non-struct)"]; nm=[["TN","FP"],["FN","TP"]]
plt.figure(figsize=(5.5,5)); plt.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
plt.xticks([0,1],labs); plt.yticks([0,1],labs); plt.xlabel("predicted"); plt.ylabel("actual")
plt.title(f"TEST confusion (final)\nacc={M['acc']:.3f} IoU={M['iou_mean']:.3f} P={M['prec']:.3f} R={M['rec']:.3f}")
for i in range(2):
    for j in range(2):
        plt.text(j,i,f"{nm[i][j]}\n{cmn[i,j]*100:.1f}%\n{int(cm[i,j]):,}px",ha="center",va="center",
                 color="black" if cmn[i,j]<0.6 else "white")
plt.tight_layout(); plt.show()


## Notes
- `ckpts/hypersim_es/`: `best.pth` (early-stop pick), `final.pth` (after Phase B), `history.json`, `split_scenes.json`.
- Early stop on **val mean-IoU** (patience 7). Change `MAX_EPOCHS`, `PATIENCE`, `PHASE_B_EPOCHS` in §0.
- TEST is evaluated once at the end — the only unbiased number. Compare **best vs final** to see if folding
  val in (Phase B) actually helped.
- This split is independent of the earlier 30-epoch run, so numbers aren't directly comparable to `trainer.ipynb`.